### Import necessary libraries

In [ ]:
import mdtraj as md
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
from sklearn.cluster import KMeans
%matplotlib inline

## Features Extraction

This section outlines the process of extracting relevant features from the molecular dynamics simulation data of the protein. Features like dihedral angles, number of hydrogen bonds, RMSD, Rg, and SASA are crucial descriptors of the protein's conformational states. By extracting these features, we can gain insights into the structural and dynamic properties of the protein over time 

## Hydrogen Bond

Tracking hydrogen bonds over time can reveal insights into the protein's folding and stability.

In [ ]:
# Identify hydrogen bonds using the wernet_nilsson method
hbonds = md.wernet_nilsson(traj)

# Count the number of hydrogen bonds in each frame
hbond_counts = [len(frame_hbonds) for frame_hbonds in hbonds]

# Create DataFrame for hydrogen bonds
hbond_df = pd.DataFrame(hbond_counts)

# Save hydrogen bond counts to a CSV file
hbond_df.to_csv('hbond_counts.csv', index=False)

# Plot the number of hydrogen bonds in each frame
plt.figure(figsize=(10, 6))
plt.plot(hbond_df.index, hbond_df[0], label='Hydrogen Bonds', color='blue')
plt.xlabel('Frame')
plt.ylabel('Number of Hydrogen Bonds')
plt.title('Number of Hydrogen Bonds in Each Frame')
plt.legend()
plt.grid(True)
plt.savefig('hbond_plot.png')
plt.show()


In [ ]:
hbond_df.head()

## Radius of Gyration

Radius of gyration (Rg) measures the compactness of the protein structure by calculating the root mean square distance of atoms from their center of mass. In this section, Rg will be computed to analyze how the compactness of the protein changes during the simulation. Fluctuations in Rg can indicate folding and unfolding events or transitions between different conformational states.

In [ ]:
# Compute the radius of gyration for each frame
radii_of_gyration = md.compute_rg(traj)

# Create DataFrame for radius of gyration
rg_df = pd.DataFrame(radii_of_gyration)

# Save the radius of gyration values to a CSV file
rg_df.to_csv('Radius_Of_Gyration.csv', index=False)

# Plot the radius of gyration over time
plt.figure(figsize=(10, 6))
plt.plot(rg_df.index, rg_df[0], label='Radius of Gyration', color='blue')
plt.xlabel('Frame')
plt.ylabel('Radius of Gyration (nm)')
plt.title('Radius of Gyration Over Time')
plt.legend()
plt.grid(True)
plt.savefig('radius_of_gyration_plot.png')
plt.show()

In [ ]:
rg_df.head()

## RMSD

RMSD measures the average deviation of atomic positions in a protein over time relative to a reference structure. This section calculates the RMSD of the protein's backbone atoms to evaluate its structural stability during the simulation. A high RMSD might indicate significant conformational changes, while a low RMSD suggests a stable structure.


In [ ]:
# Calculate RMSD for all frames relative to the first frame
rmsds = md.rmsd(traj, traj, 0)

# Create DataFrame for RMSD
rmsd_df = pd.DataFrame(rmsds)

# Save RMSD values to a CSV file
rmsd_df.to_csv('Rmsd_values.csv', index=False)

# Plot RMSD|
plt.figure(figsize=(10, 6))
plt.plot(rmsd_df.index, rmsd_df[0], label='RMSD')
plt.xlabel('Frame')
plt.ylabel('RMSD (nm)')
plt.legend()
plt.title('RMSD of Residues 40 to 56')
plt.savefig('rmsd_plot.png')
plt.show()

In [ ]:
rmsd_df.head()

## SASA

SASA is a measure of the surface area of a protein that is accessible to the solvent (e.g., water molecules). This metric provides insights into the protein’s folding state, as buried residues typically indicate a folded structure, whereas exposed residues suggest unfolding. In this section, the SASA of the protein will be computed throughout the simulation to understand how solvent exposure changes with different conformational states.

In [ ]:
# Compute the solvent accessible surface area (SASA) for each frame
sasa = md.shrake_rupley(traj, probe_radius=0.14)  # Default probe_radius for water is 0.14 nm

# Sum SASA of all atoms to get the total SASA per frame
total_sasa_per_frame = np.sum(sasa, axis=1)

# Create DataFrame for SASA
sasa_df = pd.DataFrame(total_sasa_per_frame)

# Save the total SASA values to a CSV file
sasa_df.to_csv('sasa_values.csv', index=False)

# Plot the total SASA over time
plt.figure(figsize=(10, 6))
plt.plot(sasa_df.index, sasa_df[0], label='Total SASA', color='blue')
plt.xlabel('Frame')
plt.ylabel('Total SASA (nm^2)')
plt.title('Solvent Accessible Surface Area (SASA) Over Time')
plt.legend()
plt.grid(True)
plt.savefig('sasa_plot.png')
plt.show()

In [ ]:
sasa_df.head()

## end-to-end distance 

This script calculates the end-to-end distance between two Cα atoms (residues first and last) in a single molecular dynamics trajectory. 

In [ ]:
end_to_end_distances = []

# Load the trajectory
u = mda.Universe(topology, trajectory)

# Select the first and last Cα atoms 
start_atom = u.select_atoms('name CA and resid x1')
end_atom = u.select_atoms('name CA and resid xn')

# Ensure the selected atoms are valid 
if len(start_atom) == 0 or len(end_atom) == 0:
    print("Warning: Missing start or end atom in the trajectory. Exiting.")
else:
    for ts in u.trajectory:
        start_position = start_atom.positions[0]
        end_position = end_atom.positions[0]
        end_to_end_distance = np.linalg.norm(end_position - start_position)
        end_to_end_distances.append([ts.frame, end_to_end_distance])
    df_end_to_end = pd.DataFrame(end_to_end_distances, columns=['Frame', 'End_to_End_Distance'])

    # Save the DataFrame to a CSV file
    df_end_to_end.to_csv('end_to_end_distances_single.csv', index=False)

In [ ]:
df_end_to_end.head()

## Combined file

aggregating all the extracted features—including hydrogen bonds, RMSD, Rg, SASA, and end to end distance into a single data file. This combined dataset will serve as the input for machine learning models,enabling us to classify and analyze the conformational complexity of the GB1 protein. 

In [ ]:
# Combine all features into a single DataFrame
combined_df = pd.concat([hbond_df, rg_df, rmsd_df, sasa_df, df_end_to_end], axis=1)

# Save the combined DataFrame to a CSV file
combined_df.to_csv("combined_features.csv", index=True)
print("Combined CSV file saved as 'combined_features.csv'.")